In [ ]:
# 04_train_black_box_models.ipynb
# Black-box ML models – Software Defect Prediction
# Random Forest, Gradient Boosting, SVM, MLP

import sys
sys.path.append("../src")
from utils import DATASETS, TARGET, RANDOM_STATE, N_SPLITS


import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import copy

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score
)

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

processed_path = Path("../datasets/processed")
results_path = Path("../results/black_box_models")
results_path.mkdir(parents=True, exist_ok=True)

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

def evaluate_fold(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0)
    }

models = {
    "Random_Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Gradient_Boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=1.0,
        random_state=RANDOM_STATE
    ),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(
            kernel="rbf",
            probability=True,
            random_state=RANDOM_STATE
        ))
    ]),
    "MLP": Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPClassifier(
            hidden_layer_sizes=(100,),
            max_iter=500,
            random_state=RANDOM_STATE
        ))
    ])
}

results = []

for ds in DATASETS:
    print(f"\nDATASET: {ds}")

    dataset_path = results_path / ds
    dataset_path.mkdir(parents=True, exist_ok=True)

    data = pd.read_csv(processed_path / f"{ds}_processed.csv")
    X = data.drop(columns=[TARGET])
    y = data[TARGET]

    for model_name, model in models.items():
        print(f" → {model_name}")

        model_path = dataset_path / model_name
        model_path.mkdir(parents=True, exist_ok=True)

        fold_metrics = []
        best_model = copy.deepcopy(model)  
        best_f1 = -1

        for fold, (train_idx, test_idx) in enumerate(cv.split(X, y)):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            smote = SMOTE(random_state=RANDOM_STATE)
            resampled = smote.fit_resample(X_train, y_train)
            X_train_res, y_train_res = resampled[0], resampled[1]

            fold_model = copy.deepcopy(model)
            fold_model.fit(X_train_res, y_train_res)
            y_pred = fold_model.predict(X_test)

            fm = evaluate_fold(y_test, y_pred)
            fold_metrics.append(fm)

            if fm["F1-score"] > best_f1:
                best_f1 = fm["F1-score"]
                best_model = fold_model

        avg_metrics: dict = {
            "Dataset": ds,
            "Model": model_name,
            "Accuracy": round(float(np.mean([m["Accuracy"] for m in fold_metrics])), 4),
            "Precision": round(float(np.mean([m["Precision"] for m in fold_metrics])), 4),
            "Recall": round(float(np.mean([m["Recall"] for m in fold_metrics])), 4),
            "F1-score": round(float(np.mean([m["F1-score"] for m in fold_metrics])), 4)
        }
        results.append(avg_metrics)

        joblib.dump(best_model, model_path / "model.joblib")

results_df = pd.DataFrame(results)
results_df = results_df[
    ["Dataset", "Model", "Accuracy", "Precision", "Recall", "F1-score"]
]

results_df.to_csv(
    results_path / "black_box_models_metrics.csv",
    index=False
)

results_df
